In [0]:
dbutils.widgets.text("catalog_name", "adbsflordeoro")
dbutils.widgets.text("silver_schema", "retail_silver")
dbutils.widgets.text("gold_schema", "retail_gold")

catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

print(f"Catalog: {catalog_name}")
print(f"Silver schema: {silver_schema}")
print(f"Gold schema: {gold_schema}")

In [0]:
from pyspark.sql import functions as F

CATALOG = "adbsflordeoro"
SILVER_SCHEMA = "retail_silver"
GOLD_SCHEMA = "retail_gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{GOLD_SCHEMA}`"
)

print(f"Esquema Gold: {CATALOG}.{GOLD_SCHEMA}")

In [0]:
SOURCE_TABLE = (
    f"`{CATALOG}`.`{SILVER_SCHEMA}`.`silver_sales_enriched`"
)

sales = spark.table(SOURCE_TABLE)

required_columns = {
    "date", "store_nbr", "family", "sales",
    "onpromotion", "city", "state", "store_type",
    "cluster", "transactions"
}

missing = required_columns - set(sales.columns)

if missing:
    raise ValueError(
        f"Faltan columnas en Silver: {sorted(missing)}"
    )

print(f"Registros de entrada: {sales.count():,}")
sales.printSchema()

In [0]:
from pyspark.sql import functions as F

print("=== VALIDACIÓN DE SALES ===")

sales.select(
    F.count("*").alias("total_rows"),
    F.count("transactions").alias("transactions_non_null"),
    F.sum(
        F.when(F.col("transactions").isNull(), 1).otherwise(0)
    ).alias("transactions_null")
).show()

print("=== EJEMPLOS CON TRANSACCIONES NULL ===")

sales.filter(
    F.col("transactions").isNull()
).select(
    "date",
    "store_nbr",
    "transactions"
).distinct().show(10, truncate=False)

print("=== EJEMPLOS CON TRANSACCIONES INFORMADAS ===")

sales.filter(
    F.col("transactions").isNotNull()
).select(
    "date",
    "store_nbr",
    "transactions"
).distinct().show(10, truncate=False)

In [0]:
gold_sales_daily = (
    sales
    .groupBy(
        "date",
        "store_nbr",
        "city",
        "state",
        "store_type",
        "cluster"
    )
    .agg(
        F.sum("sales").alias("total_sales"),
        F.sum("onpromotion").alias("promotion_item_count"),
        F.max("transactions").alias("transactions"),
        F.count("*").alias("product_family_rows")
    )
    .withColumn(
        "sales_per_transaction",
        F.when(
            F.col("transactions") > 0,
            F.col("total_sales") / F.col("transactions")
        )
    )
)

(
    gold_sales_daily.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        "abfss://gold@adlsgr18smartdata.dfs.core.windows.net/gold_sales_daily"
    )
    .saveAsTable(
        f"`{CATALOG}`.`{GOLD_SCHEMA}`.`gold_sales_daily_ext`"
    )
)

print("Tabla externa gold_sales_daily_ext creada.")

In [0]:
gold_sales_by_family = (
    sales
    .groupBy(
        "date",
        "family",
        "store_nbr",
        "city",
        "state"
    )
    .agg(
        F.sum("sales").alias("total_sales"),
        F.sum("onpromotion").alias("promotion_item_count")
    )
)

(
    gold_sales_by_family.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        "abfss://gold@adlsgr18smartdata.dfs.core.windows.net/gold_sales_by_family"
    )
    .saveAsTable(
        f"`{CATALOG}`.`{GOLD_SCHEMA}`.`gold_sales_by_family_ext`"
    )
)

print("Tabla externa gold_sales_by_family_ext creada.")

In [0]:
gold_store_performance = (
    gold_sales_daily
    .groupBy(
        "store_nbr",
        "city",
        "state",
        "store_type",
        "cluster"
    )
    .agg(
        F.sum("total_sales").alias("total_sales"),
        F.sum("transactions").alias("total_transactions"),
        F.countDistinct("date").alias("active_dates"),
        F.avg("total_sales").alias("avg_daily_sales")
    )
    .withColumn(
        "sales_rank",
        F.dense_rank().over(
            __import__("pyspark").sql.Window.orderBy(
                F.col("total_sales").desc()
            )
        )
    )
)

(
    gold_store_performance.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        "abfss://gold@adlsgr18smartdata.dfs.core.windows.net/gold_store_performance"
    )
    .saveAsTable(
        f"`{CATALOG}`.`{GOLD_SCHEMA}`.`gold_store_performance_ext`"
    )
)

print("Tabla externa gold_store_performance_ext creada.")

In [0]:
gold_tables = [
    "gold_sales_daily_ext",
    "gold_sales_by_family_ext",
    "gold_store_performance_ext"
]

for table_name in gold_tables:
    full_name = (
        f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{table_name}`"
    )

    df = spark.table(full_name)

    detail = spark.sql(
        f"DESCRIBE DETAIL {full_name}"
    ).first()

    print(f"\nTabla: {table_name}")
    print(f"Filas: {df.count():,}")
    print(f"Formato: {detail['format']}")
    print(f"Ubicación: {detail['location']}")
    df.show(5, truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_transactions = spark.table(
    "adbsflordeoro.retail_silver.silver_transactions"
)

gold_daily = spark.table(
    "adbsflordeoro.retail_gold.gold_sales_daily_ext"
)

# Preparar transacciones de Silver
silver_tx = silver_transactions.select(
    "date",
    "store_nbr",
    F.col("transactions").alias("silver_transactions")
)

# Preparar transacciones de Gold
gold_tx = gold_daily.select(
    "date",
    "store_nbr",
    F.col("transactions").alias("gold_transactions")
)

# Comparación conservando todos los registros de Gold
comparison = (
    gold_tx.alias("g")
    .join(
        silver_tx.alias("s"),
        on=["date", "store_nbr"],
        how="left"
    )
    .withColumn(
        "validation_status",
        F.when(
            F.col("silver_transactions").isNull(),
            "SIN_REGISTRO_EN_SILVER"
        )
        .when(
            F.col("gold_transactions").isNull(),
            "NULL_EN_GOLD"
        )
        .when(
            F.col("silver_transactions") != F.col("gold_transactions"),
            "DIFERENCIA"
        )
        .otherwise("OK")
    )
)

# Resumen de validación
display(
    comparison
    .groupBy("validation_status")
    .count()
    .orderBy("validation_status")
)

# Mostrar únicamente registros que requieren revisión
display(
    comparison
    .filter(F.col("validation_status") != "OK")
    .orderBy("date", "store_nbr")
)

In [0]:
from pyspark.sql import functions as F

silver_transactions = spark.table(
    "adbsflordeoro.retail_silver.silver_transactions"
)

gold_daily = spark.table(
    "adbsflordeoro.retail_gold.gold_sales_daily_ext"
)

# Comparar por fecha y tienda
silver_tx = silver_transactions.select(
    "date",
    "store_nbr",
    F.col("transactions").alias("silver_transactions")
)

gold_tx = gold_daily.select(
    "date",
    "store_nbr",
    F.col("transactions").alias("gold_transactions")
)

comparison = (
    silver_tx
    .join(
        gold_tx,
        on=["date", "store_nbr"],
        how="full"
    )
    .withColumn(
        "matches",
        F.when(
            F.col("silver_transactions").isNull()
            | F.col("gold_transactions").isNull(),
            F.lit(False)
        ).otherwise(
            F.col("silver_transactions")
            == F.col("gold_transactions")
        )
    )
)

display(
    comparison.filter(~F.col("matches"))
)

In [0]:
gold_tables = [
    "adbsflordeoro.retail_gold.gold_sales_daily",
    "adbsflordeoro.retail_gold.gold_sales_by_family",
    "adbsflordeoro.retail_gold.gold_store_performance"
]

for table in gold_tables:
    print(f"\n--- {table} ---")

    spark.sql(f"DESCRIBE DETAIL {table}") \
        .select(
            "name",
            "format",
            "location",
            "numFiles",
            "sizeInBytes"
        ) \
        .show(truncate=False)

In [0]:
%sql SHOW TABLES IN adbsflordeoro.retail_gold;

In [0]:
gold_tables = [
    "gold_sales_daily_ext",
    "gold_sales_by_family_ext",
    "gold_store_performance_ext"
]

for table in gold_tables:
    full_name = f"adbsflordeoro.retail_gold.{table}"

    print(f"\n--- {full_name} ---")

    spark.sql(f"DESCRIBE DETAIL {full_name}") \
        .select("name", "location", "numFiles", "sizeInBytes") \
        .show(truncate=False)

In [0]:
%sql
SELECT 'gold_sales_daily_ext' AS table_name, COUNT(*) AS total_rows
FROM adbsflordeoro.retail_gold.gold_sales_daily_ext

UNION ALL

SELECT 'gold_sales_by_family_ext', COUNT(*)
FROM adbsflordeoro.retail_gold.gold_sales_by_family_ext

UNION ALL

SELECT 'gold_store_performance_ext', COUNT(*)
FROM adbsflordeoro.retail_gold.gold_store_performance_ext

UNION ALL

SELECT 'silver_sales_enriched', COUNT(*)
FROM adbsflordeoro.retail_silver.silver_sales_enriched;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT family) AS distinct_families
FROM adbsflordeoro.retail_gold.gold_sales_by_family_ext;

In [0]:
%sql
SELECT *
FROM adbsflordeoro.retail_gold.gold_sales_by_family_ext
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date) AS total_dates,
    COUNT(DISTINCT family) AS total_families,
    COUNT(DISTINCT store_nbr) AS total_stores,
    SUM(total_sales) AS total_sales,
    SUM(promotion_item_count) AS total_promotions
FROM adbsflordeoro.retail_gold.gold_sales_by_family_ext;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date) AS total_dates,
    SUM(total_sales) AS total_sales
FROM adbsflordeoro.retail_gold.gold_sales_daily_ext;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT store_nbr) AS total_stores,
    SUM(total_sales) AS total_sales
FROM adbsflordeoro.retail_gold.gold_store_performance_ext;

In [0]:
from pyspark.sql import functions as F

silver_transactions = spark.table(
    "adbsflordeoro.retail_silver.silver_transactions"
)

gold_daily = spark.table(
    "adbsflordeoro.retail_gold.gold_sales_daily_ext"
)

print("=== SILVER TRANSACTIONS ===")
silver_transactions.select(
    F.count("*").alias("total_rows"),
    F.count("transactions").alias("non_null_transactions"),
    F.sum(
        F.when(F.col("transactions").isNull(), 1).otherwise(0)
    ).alias("null_transactions")
).show()

print("=== GOLD SALES DAILY ===")
gold_daily.select(
    F.count("*").alias("total_rows"),
    F.count("transactions").alias("non_null_transactions"),
    F.sum(
        F.when(F.col("transactions").isNull(), 1).otherwise(0)
    ).alias("null_transactions")
).show()

print("=== GOLD DAILY SAMPLE ===")
gold_daily.select(
    "date", "store_nbr", "transactions"
).show(10, truncate=False)